In [1]:
import os
os.chdir('../../../../../..')

In [2]:
import numpy as np
import hdbscan
import polars as pl
import matplotlib.pyplot as plt
import seaborn as sns
from rdkit import Chem
import gc
import numpy as np
import polars as pl
from dscribe.descriptors import SOAP
from rdkit import Chem
import numpy as np
import polars as pl
from rdkit import Chem
from rdkit.Chem import AllChem
import numpy as np
import polars as pl
from ase import Atoms
from dscribe.descriptors import SOAP
from rdkit import Chem
import lightgbm as lgb
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.model_selection import train_test_split
import numpy as np
import pandas as pd
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler




from sklearn.cluster import AgglomerativeClustering, SpectralClustering, DBSCAN
from kmedoids import KMedoids
from scipy.cluster.hierarchy import dendrogram, linkage
from scipy.spatial.distance import squareform
from umap import UMAP
from dscribe.kernels import REMatchKernel

from src.datasets import QM9Dataset
from src.helper_functions import plot_molecules_with_py3dmol, create_chemiscope_viewer, plot_distance_matrix_projection, evaluate_distance_matrix_clustering_sweep, average_numeric_by_cluster, evaluate_hdbscan_grid, get_isomers

projection_method = "MDS"

In [3]:
qm9 = QM9Dataset(limit=10_000, descriptors=["mace", "soap"])
df = qm9.load()

2026-05-22 13:29:17.481 | INFO     | src.datasets:_load_full_qm9_df:817 - Loading cached full QM9 dataset from: data/QM9/dataset_cleaned.parquet
2026-05-22 13:29:17.851 | INFO     | src.datasets:_sample_qm9_df:1000 - QM9 sampling complete: strategy=stratified, requested_limit=10000, returned_rows=10000, sampling on columns=['num_atoms', 'gap'].
2026-05-22 13:29:17.851 | INFO     | src.datasets:_add_requested_descriptors:202 - Applying requested QM9 descriptors to sampled dataframe (rows=10000).
2026-05-22 13:29:17.852 | INFO     | src.features:compute_mace_outputs:679 - Computing MACE embeddings (model=medium, batch_size=32)...
/Users/karlfindhansen/thesis/Anomaly-Detection-in-Molecular-and-Materials-Datasets/.venv/lib/python3.12/site-packages/e3nn/o3/_wigner.py:10: UserWarning: Environment variable TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD detected, since the`weights_only` argument was not explicitly passed to `torch.load`, forcing weights_only=False.
  _Jd, _W3j_flat, _W3j_indices = torch.loa

cuequivariance or cuequivariance_torch is not available. Cuequivariance acceleration will be disabled.
Using MACE-OFF23 MODEL for MACECalculator with /Users/karlfindhansen/.cache/mace/MACE-OFF23_medium.model
Using float32 for MACECalculator, which is faster but less accurate. Recommended for MD. Use float64 for geometry optimization.


2026-05-22 13:36:59.899 | SUCCESS  | src.datasets:add_mace:1236 - Added MACE embeddings and matrices.
2026-05-22 13:36:59.911 | INFO     | src.features:compute_soap_outputs:395 - Computing SOAP (rcut=6.0, nmax=8, lmax=6, normalize=False)...
2026-05-22 13:37:06.529 | SUCCESS  | src.datasets:add_soap:1193 - Added SOAP embeddings and matrices.
2026-05-22 13:37:06.529 | INFO     | src.datasets:_add_requested_descriptors:213 - Added descriptor column(s): ['mace_embedding', 'mace_matrix', 'soap_embedding', 'soap_matrix']
2026-05-22 13:37:06.536 | INFO     | src.datasets:_load_with_descriptor_filter:857 - QM9 descriptor null-filtering complete: attempts=1, requested_limit=10000, returned_rows=10000, base_rows=10000.


# Hypothesis 1
- mace will be better to predict the dipole moment of isomers. because it is trained to understand the higher order geometries of a molecule. MACE uses equivariant message passing. In each layer, atoms pass directional, geometric information to their neighbors. Over multiple layers, this allows MACE to build a high-body-order, global topological map of the molecule. It natively tracks how a change on one side of the molecule affects the electronic asymmetry on the other side, allowing it to map the shifting dipole moments precisely.

In [4]:
import numpy as np
import polars as pl
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.model_selection import KFold
from sklearn.preprocessing import StandardScaler

# -------------------------------------------------------------------------
# 1. ISOLATE ISOMER SUBSET
# -------------------------------------------------------------------------
def isolate_top_isomers(df: pl.DataFrame, formula_col: str = "chemical_formula", min_isomers: int = 30):
    formula_counts = (
        df.group_by(formula_col)
        .agg(pl.len().alias("count"))
        .filter(pl.col("count") >= min_isomers)
    )
    if formula_counts.is_empty():
        raise ValueError(f"No formulas found with at least {min_isomers} isomers.")
    
    return df.filter(pl.col(formula_col).is_in(formula_counts[formula_col].to_list()))

# -------------------------------------------------------------------------
# 2. REGRESSION PIPELINE
# -------------------------------------------------------------------------
def evaluate_isomer_regression(df: pl.DataFrame, embedding_col: str, property_col: str, n_splits: int = 5):
    X = np.vstack(df[embedding_col].to_numpy())
    y = df[property_col].to_numpy()
    
    kf = KFold(n_splits=n_splits, shuffle=True, random_state=42)
    maes = []
    r2s = []
    
    for train_idx, test_idx in kf.split(X, y):
        X_train, X_test = X[train_idx], X[test_idx]
        y_train, y_test = y[train_idx], y[test_idx]
        
        scaler = StandardScaler()
        X_train_scaled = scaler.fit_transform(X_train)
        X_test_scaled = scaler.transform(X_test)
        
        # Ridge regression acts as our linear probe
        reg = Ridge(alpha=1.0)
        reg.fit(X_train_scaled, y_train)
        
        preds = reg.predict(X_test_scaled)
        maes.append(mean_absolute_error(y_test, preds))
        r2s.append(r2_score(y_test, preds))
        
    return np.mean(maes), np.std(maes), np.mean(r2s)

# Filter down so the model is forced to learn property variances purely from layout differences
isomer_subset = isolate_top_isomers(df, formula_col="formula", min_isomers=30)
isomer_subset = df

print(f"Evaluating dipole regression across {len(isomer_subset)} closely matched isomers...\n")

soap_mae, soap_std, soap_r2 = evaluate_isomer_regression(isomer_subset, "soap_embedding", "mu")
print(f"SOAP -> Dipole MAE: {soap_mae:.4f} ± {soap_std:.4f} | R² Score: {soap_r2:.4f}")

mace_mae, mace_std, mace_r2 = evaluate_isomer_regression(isomer_subset, "mace_embedding", "mu")
print(f"MACE -> Dipole MAE: {mace_mae:.4f} ± {mace_std:.4f} | R² Score: {mace_r2:.4f}")

Evaluating dipole regression across 10000 closely matched isomers...

SOAP -> Dipole MAE: 1.0257 ± 0.0212 | R² Score: 0.1980
MACE -> Dipole MAE: 0.7098 ± 0.0079 | R² Score: 0.6044


In [5]:
print(df.columns)

['mol_id', 'formula', 'smiles', 'canonical_smiles', 'scaffold_smiles', 'generic_scaffold', 'root_scaffold', 'brics_fragments', 'scaffold_tree_nodes', 'selfies', 'functional_groups', 'structure_class', 'is_injected', 'outlier_category', 'mol_weight', 'logp', 'tpsa', 'election_affinity', 'ionization_energies', 'num_heavy_atoms', 'num_rings', 'num_aromatic_rings', 'num_fluorine', 'num_heteroatoms', 'num_atoms', 'coordination', 'num_rotatable_bonds', 'fraction_csp1', 'fraction_csp2', 'fraction_csp3', 'h_bond_donors', 'h_bond_acceptors', 'branching_index', 'num_sp_carbons', 'num_sp2_carbons', 'num_sp3_carbons', 'main_chain_length', 'raw_token_count', 'avg_bond_length', 'fr_benzene', 'fr_alcohol', 'fr_phenol', 'fr_amine', 'fr_amide', 'fr_carboxylic_acid', 'fr_ester', 'fr_ketone', 'fr_ether', 'fr_nitro', 'coordinates', 'atomic_numbers', 'mu', 'alpha', 'homo', 'lumo', 'gap', 'r2', 'zpve', 'u0', 'u', 'h', 'g', 'cv', 'u0_atom', 'u_atom', 'h_atom', 'g_atom', 'A', 'B', 'C', 'mace_embedding', 'mace

# Hypothesis 2

In [6]:
def run_linear_probe(df, embedding_col, target_col):
    X = np.vstack(df[embedding_col].to_numpy())
    y = df[target_col].to_numpy()
    
    # Filter out classes with very few samples to ensure clean cross-validation
    unique, counts = np.unique(y, return_counts=True)
    valid_classes = unique[counts >= 10]
    
    # Filter matrix and labels
    mask = np.isin(y, valid_classes)
    X, y = X[mask], y[mask]
    
    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    accuracies = []
    
    for train_idx, test_idx in skf.split(X, y):
        X_train, X_test = X[train_idx], X[test_idx]
        y_train, y_test = y[train_idx], y[test_idx]
        
        scaler = StandardScaler()
        X_train_scaled = scaler.fit_transform(X_train)
        X_test_scaled = scaler.transform(X_test)
        
        # Linear probe (No hidden layers, purely tests linear separability)
        model = LogisticRegression(max_iter=1000, random_state=42)
        model.fit(X_train_scaled, y_train)
        
        preds = model.predict(X_test_scaled)
        accuracies.append(accuracy_score(y_test, preds))
        
    return np.mean(accuracies), np.std(accuracies)

# --- Evaluate both spaces on "num_rings" ---
print("Evaluating Linear Separability of Topological Concepts...")

mace_acc, mace_std = run_linear_probe(df, "mace_embedding", "num_rings")
print(f"MACE -> Ring Classification Accuracy: {mace_acc:.4f} ± {mace_std:.4f}")

soap_acc, soap_std = run_linear_probe(df, "soap_embedding", "num_rings")
print(f"SOAP -> Ring Classification Accuracy: {soap_acc:.4f} ± {soap_std:.4f}")

Evaluating Linear Separability of Topological Concepts...
MACE -> Ring Classification Accuracy: 0.9851 ± 0.0016
SOAP -> Ring Classification Accuracy: 0.8399 ± 0.0063


# Hypothesis 2

In [7]:
...

Ellipsis